In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.colors import LinearSegmentedColormap

# =========================
# Colors
# =========================
OFF_BLACK = "#1E1E1E"
COOL_GREY = "#8F98A8"
LIGHT_GREY = "#EEF1F5"
SATURATED_BLUE = "#245BFF"
WHITE = "#FFFFFF"

np.random.seed(42)

# =========================
# Fixed GNN structure
# =========================
num_nodes = 15
radius = 0.34

G = nx.random_geometric_graph(num_nodes, radius=radius, seed=42)

while not nx.is_connected(G):
    G = nx.random_geometric_graph(
        num_nodes,
        radius=radius,
        seed=np.random.randint(10000)
    )

pos = nx.get_node_attributes(G, "pos")

# Rescale positions so nodes do not get cut
margin = 0.08
xs = np.array([p[0] for p in pos.values()])
ys = np.array([p[1] for p in pos.values()])

for n, (x, y) in pos.items():
    x_new = margin + (x - xs.min()) / (xs.max() - xs.min()) * (1 - 2 * margin)
    y_new = margin + (y - ys.min()) / (ys.max() - ys.min()) * (1 - 2 * margin)
    pos[n] = (x_new, y_new)

# Fixed node scores
scores = np.random.rand(num_nodes)
top_k = 4
top_nodes = set(np.argsort(scores)[-top_k:])

# =========================
# Shared visual settings
# =========================
node_colors = [
    SATURATED_BLUE if n in top_nodes else LIGHT_GREY
    for n in G.nodes()
]

node_sizes = [
    260 if n in top_nodes else 165
    for n in G.nodes()
]

edge_colors = [
    SATURATED_BLUE if (u in top_nodes or v in top_nodes) else COOL_GREY
    for u, v in G.edges()
]

edge_widths = [
    1.8 if (u in top_nodes or v in top_nodes) else 1.0
    for u, v in G.edges()
]

edge_alphas = [
    0.50 if (u in top_nodes or v in top_nodes) else 0.42
    for u, v in G.edges()
]

# 6 x 6 at 300 dpi = 1800 x 1800 px
FIGSIZE = (6, 6)
DPI = 300


# =========================
# Function 1: GNN only
# =========================
def plot_gnn_only():
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    fig.patch.set_facecolor(WHITE)
    ax.set_facecolor(WHITE)

    for (u, v), color, width, alpha in zip(
        G.edges(), edge_colors, edge_widths, edge_alphas
    ):
        nx.draw_networkx_edges(
            G,
            pos,
            edgelist=[(u, v)],
            ax=ax,
            edge_color=color,
            width=width,
            alpha=alpha,
        )

    nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_size=node_sizes,
        node_color=node_colors,
        edgecolors=OFF_BLACK,
        linewidths=0.9,
        alpha=0.98,
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")

    plt.savefig(
        "01_fixed_gnn_structure.png",
        dpi=DPI,
        bbox_inches="tight",
        pad_inches=0.03
    )
    plt.show()


# =========================
# Function 2: LUR grid + same GNN
# =========================
def plot_lur_grid_overlay():
    grid_size = 55
    x = np.linspace(0, 1, grid_size)
    y = np.linspace(0, 1, grid_size)
    xx, yy = np.meshgrid(x, y)

    grid_score = np.zeros_like(xx)

    # Local ST-GNN node influence
    sigma_node = 0.085

    for node, (px, py) in pos.items():
        if node in top_nodes:
            weight = 2.3 * scores[node]
        else:
            weight = 0.15 * scores[node]

        dist2 = (xx - px) ** 2 + (yy - py) ** 2
        grid_score += weight * np.exp(-dist2 / (2 * sigma_node ** 2))

    # =========================
    # Broader LUR-like spatial trends
    # These create an overall blue spatial pattern
    # independent from the selected blue GNN nodes.
    # =========================

    # Large west-to-east / southwest-to-northeast gradient
    broad_gradient = 0.75 * (
        0.45 * xx + 0.35 * (1 - yy) + 0.20 * np.sin(2.5 * np.pi * xx)
    )

    # Main diagonal urban / traffic corridor
    corridor_1 = 0.90 * np.exp(
        -((yy - (0.18 + 0.58 * xx)) ** 2) / (2 * 0.09 ** 2)
    )

    # Secondary curved corridor
    corridor_2 = 0.55 * np.exp(
        -((yy - (0.78 - 0.35 * xx + 0.08 * np.sin(2 * np.pi * xx))) ** 2)
        / (2 * 0.11 ** 2)
    )

    # Broad regional hotspots
    regional_hotspot_1 = 0.70 * np.exp(
        -((xx - 0.72) ** 2 + (yy - 0.35) ** 2) / (2 * 0.30 ** 2)
    )

    regional_hotspot_2 = 0.55 * np.exp(
        -((xx - 0.25) ** 2 + (yy - 0.68) ** 2) / (2 * 0.34 ** 2)
    )

    regional_hotspot_3 = 0.45 * np.exp(
        -((xx - 0.52) ** 2 + (yy - 0.52) ** 2) / (2 * 0.42 ** 2)
    )

    grid_score += (
        broad_gradient
        + corridor_1
        + corridor_2
        + regional_hotspot_1
        + regional_hotspot_2
        + regional_hotspot_3
    )

    # Normalize and increase contrast
    grid_score = (grid_score - grid_score.min()) / (
        grid_score.max() - grid_score.min()
    )

    # Lower exponent makes broader blue trends more visible
    grid_score = grid_score ** 0.62

    cmap = LinearSegmentedColormap.from_list(
        "poster_blue_grid",
        [
            WHITE,
            "#F0F4FF",
            "#D9E4FF",
            "#AEC4FF",
            "#6F91FF",
            SATURATED_BLUE,
        ],
    )

    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    fig.patch.set_facecolor(WHITE)
    ax.set_facecolor(WHITE)

    ax.imshow(
        grid_score,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap=cmap,
        alpha=0.96,
        interpolation="nearest",
    )

    # Subtle grid lines
    for gx in x:
        ax.plot([gx, gx], [0, 1], color=WHITE, linewidth=0.25, alpha=0.38)

    for gy in y:
        ax.plot([0, 1], [gy, gy], color=WHITE, linewidth=0.25, alpha=0.38)

    # Same GNN overlay
    for (u, v), color, width, alpha in zip(
        G.edges(), edge_colors, edge_widths, edge_alphas
    ):
        nx.draw_networkx_edges(
            G,
            pos,
            edgelist=[(u, v)],
            ax=ax,
            edge_color=color,
            width=width,
            alpha=alpha,
        )

    nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_size=node_sizes,
        node_color=node_colors,
        edgecolors=OFF_BLACK,
        linewidths=0.9,
        alpha=0.98,
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")

    plt.savefig(
        "02_lur_grid_overlay_same_gnn_structure.png",
        dpi=DPI,
        bbox_inches="tight",
        pad_inches=0.03
    )
    plt.show()


plot_gnn_only()
plot_lur_grid_overlay()